# Water Quality Potability Prediction

This notebook explores the supplied water-quality dataset and compares Decision Tree and K-Nearest Neighbors (KNN) classifiers. It uses pipelines to keep preprocessing inside cross-validation and avoid leakage from the test set.

> **Safety note:** This is an educational model, not a certified method for determining whether water is safe to drink.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

# Works when the notebook is run from the repository root or from notebooks/.
ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "data" / "water_potability.csv").exists():
    ROOT_DIR = ROOT_DIR.parent
DATA_PATH = ROOT_DIR / "data" / "water_potability.csv"

## 2. Load and inspect the data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())

In [ ]:
display(df.info())
display(df.describe().T)
print("Missing values per column:")
display(df.isna().sum().to_frame("missing_values"))

## 3. Exploratory data analysis

In [ ]:
ax = df["Potability"].value_counts().sort_index().plot(kind="bar", rot=0)
ax.set(title="Potability class distribution", xlabel="Potability", ylabel="Number of samples")
plt.tight_layout()
plt.show()

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if not missing.empty:
    ax = missing.plot(kind="bar", rot=30)
    ax.set(title="Missing values by feature", xlabel="Feature", ylabel="Missing values")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found.")

In [ ]:
_ = df.hist(figsize=(14, 10), bins=25, edgecolor="black")
plt.suptitle("Feature distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Feature correlation matrix")
plt.tight_layout()
plt.show()

## 4. Prepare the train/test split

In [ ]:
TARGET = "Potability"
X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training samples: {len(X_train):,}")
print(f"Test samples: {len(X_test):,}")
print("Training target proportions:")
display(y_train.value_counts(normalize=True).sort_index().rename("proportion"))

## 5. Define preprocessing and cross-validation

Missing numeric values are imputed with the median inside each pipeline. This means each cross-validation fold learns imputation values only from its own training portion. KNN also scales the features because distance-based models are sensitive to feature scale.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Decision Trees do not require feature scaling.
dt_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
])

# KNN is distance-based, so standardize features after imputation.
knn_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier()),
])

## 6. Tune the Decision Tree

In [ ]:
dt_param_grid = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [None, 5, 10, 20],
    "model__min_samples_split": [2, 10],
}

dt_search = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=dt_param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
)
dt_search.fit(X_train, y_train)
print("Best Decision Tree parameters:", dt_search.best_params_)
print(f"Best cross-validation F1: {dt_search.best_score_:.3f}")

## 7. Tune K-Nearest Neighbors

In [ ]:
knn_param_grid = {
    "model__n_neighbors": [5, 11, 21],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan"],
}

knn_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
)
knn_search.fit(X_train, y_train)
print("Best KNN parameters:", knn_search.best_params_)
print(f"Best cross-validation F1: {knn_search.best_score_:.3f}")

## 8. Evaluate on the held-out test set

In [ ]:
models = {
    "Decision Tree": dt_search.best_estimator_,
    "KNN": knn_search.best_estimator_,
}
results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (class 1)": report["1"]["precision"],
        "Recall (class 1)": report["1"]["recall"],
        "F1 (class 1)": report["1"]["f1-score"],
    })
    print(f"\n{name} classification report")
    print(classification_report(y_test, y_pred, zero_division=0))

results_df = pd.DataFrame(results).set_index("Model").sort_values("F1 (class 1)", ascending=False)
display(results_df.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 9. Example prediction

In [ ]:
# Example: use the first row of the held-out test data.
# The predicted label follows the dataset's Potability encoding (0 or 1).
sample = X_test.iloc[[0]]
print("Predicted potability label:", int(dt_search.best_estimator_.predict(sample)[0]))
print("Actual label:", int(y_test.iloc[0]))

## 10. Conclusions and limitations

- Compare the models using the held-out metrics rather than accuracy alone.
- The data may contain measurement and sampling limitations that are not assessed here.
- This project is for learning and experimentation. Do not use its predictions as the sole basis for determining whether water is safe to drink; use appropriate laboratory testing and public-health guidance.